# Notebook 01: DeepSAM: the model, the three elements, and training the surplus network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yangycpku/Machine_Learning_Macro_PSU/blob/main/Tutorials/Tutorial_DeepSAM/notebooks/01_DeepSAM_Model_Loss_Training.ipynb)

**Course:** Penn State Mini-Course on Deep Learning and Heterogeneous Agent Macroeconomics (Penn State University, September 15, 2026)
**Session:** Lecture 2 tutorial: Deep Learning for Continuous Time Models and Structural Estimation (DeepSAM)
**Slides:** [`Lectures/Lecture2_slides_Continuous_Time_Structural_Estimation.pdf`](https://github.com/yangycpku/Machine_Learning_Macro_PSU/blob/main/Lectures/Lecture2_slides_Continuous_Time_Structural_Estimation.pdf)
**Notebook role:** core (in-class walkthrough)
**Runtime:** ~3 min at `smoke` on an A100, ~15 min at `teaching`
**Author:** Yucheng Yang (University of Zurich and Swiss Finance Institute). [Course repository](https://github.com/yangycpku/Machine_Learning_Macro_PSU)

---

This notebook is the DeepSAM method end to end on the Section 3 economy of *Deep Learning
for Search and Matching Models*: a labour search-and-matching model with **two-sided
heterogeneity** (worker types $x$, firm types $y$), **aggregate shocks**, and
**distributional feedback**: the distribution of existing matches changes the value of
forming new ones.

After the model (Sections I–V, the write-up from the replication package), the code follows
the **three elements** of a deep-learning solution method from the lecture:

1. **Parameterize the unknown function.** The match surplus $S(x, y, z, g)$ becomes a neural
   network that takes the 55-dimensional distribution $g$ as a direct input.
2. **Specify the objective.** The residual of the master equation, coded term by term.
3. **Choose where to evaluate it.** Sampling the state $(x, y, z, g)$: first mixtures of
   steady states, then the ergodic set of the simulated economy.

With the three elements in place, the last part trains the network (briefly) and shows how
well the trained one satisfies the equation across type space. Notebook 02 then uses the
solved model for the COVID experiment.

In [ ]:
RUN_MODE = "smoke"     # one of: "smoke", "teaching", "production"

## Set up the code directory

`src/train_nn.py` holds the whole method: the deterministic steady states, the neural
network, the master-equation residual, the simulation of the distribution, and the training
loop. It expects to be imported with the project root (`Tutorials/Tutorial_DeepSAM`) as the
working directory, because `solve_steady_state` writes its output there as `.npy` files.

* **Google Colab** (the default for this course). The first run clones the course repository
  into `/content` and moves into `Tutorials/Tutorial_DeepSAM`. Colab already ships PyTorch,
  NumPy, SciPy, matplotlib and OmegaConf, so nothing needs to be installed. Use a **GPU
  runtime** (`Runtime -> Change runtime type`): the timings below are for an A100.
* **A local clone.** Open the notebook from inside `Tutorials/Tutorial_DeepSAM/notebooks` and
  the cell steps up to the project root.

The cell also checks that PyTorch can actually launch a kernel on the runtime's GPU. If the
check fails, the GPU is hidden and everything runs on the CPU (slowly). Set
`FORCE_CPU = True` to do that unconditionally. Run this cell **before** any cell that
imports `torch`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/yangycpku/Machine_Learning_Macro_PSU.git"
REPO_DIR = "/content/Machine_Learning_Macro_PSU"
PROJ_DIR = os.path.join(REPO_DIR, "Tutorials", "Tutorial_DeepSAM")
FORCE_CPU = False   # set True to ignore any GPU and run PyTorch on the CPU

try:
    import google.colab  # noqa: F401  (importable only on a Colab runtime)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isfile(os.path.join(PROJ_DIR, "src", "train_nn.py")):
        print("Cloning the course repository into", REPO_DIR, "...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(PROJ_DIR)
elif os.path.isfile("../src/train_nn.py"):
    os.chdir("..")           # a local clone: the notebook lives in notebooks/

if not (os.path.isfile("src/train_nn.py") and os.path.isfile("config/config.yaml")):
    raise FileNotFoundError(
        f"Expected the DeepSAM project root (Tutorials/Tutorial_DeepSAM), but the working "
        f"directory is {os.getcwd()!r}. Open this notebook from inside notebooks/, or "
        f"os.chdir() to the project root."
    )

ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# GPU check, in a separate process so a failure cannot poison this kernel. If PyTorch
# cannot launch a kernel on the GPU, the GPU is hidden and everything runs on the CPU.
_PROBE = ("import torch; x = torch.randn(8, 8, device='cuda'); "
          "print(torch.cuda.get_device_name(0)); (x @ x).sum().item()")
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("FORCE_CPU = True: PyTorch will run on the CPU.")
elif "torch" in sys.modules:
    print("torch is already imported, so the GPU check was skipped "
          "(restart the runtime to run it again).")
else:
    _r = subprocess.run([sys.executable, "-c", _PROBE], capture_output=True, text=True)
    if _r.returncode == 0:
        print("GPU check passed:", _r.stdout.strip().split("\n")[0])
    else:
        _last = [l for l in _r.stderr.strip().split("\n") if l.strip()]
        _last = _last[-1][:160] if _last else "(no error text)"
        if "no CUDA" in _last or "Torch not compiled" in _last or "cuda" in _last.lower() and "available" in _last.lower():
            print("No GPU in this runtime: PyTorch will run on the CPU (slow; use a GPU runtime).")
        else:
            os.environ["CUDA_VISIBLE_DEVICES"] = ""
            print("GPU check FAILED, so the GPU is hidden and PyTorch will run on the CPU.\n  Error was:", _last)

try:
    import omegaconf  # noqa: F401  (preinstalled on Colab; installed here if missing)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omegaconf"], check=True)

print("Running on Colab:", IN_COLAB)
print("Project root:", ROOT)

In [ ]:
import contextlib
import inspect
import io
import random
import time
import warnings

import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from omegaconf import OmegaConf

from train_nn import Train_NN, Master_PINN_S
import calibration_plot as calplot
import covid_shock_plot as covplot
import plotting

# Two harmless notices from the library and from PyTorch's autograd, silenced for readability.
warnings.filterwarnings("ignore", message=".*To copy construct from a tensor.*")
warnings.filterwarnings("ignore", message=".*no current CUDA context.*")
warnings.filterwarnings("ignore", message=".*pin_memory.*")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
else:
    print("CPU (no GPU visible) -- everything below still runs, but slowly")

## Choose the run mode

The expensive things in this notebook are the simulation that builds the training pool and
the number of gradient steps. `RUN_MODE` maps onto both.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| ergodic-pool paths × horizon | 32 × 500 | 64 × 1000 | 256 × 5000 |
| training steps (each of two runs) | 500 | 5,000 | 20,000 |
| wall clock on an A100 | ~3 min | ~15 min | ~1 h |

`production` matches the settings in the replication package. Note what is *not* on this
list: training the surplus network to convergence. The full pipeline behind the shipped
checkpoint is a homotopy initialisation, a long main training phase run to a loss
threshold, and then 8 further rounds of 100,000 gradient steps with the ergodic dataset
rebuilt between rounds, several hours on an A100. The training section below therefore
quantifies the gap rather than trying to close it.

In [ ]:
if RUN_MODE == "smoke":
    SIM_PATHS, SIM_T = 32, 500
    TRAIN_STEPS = 500
elif RUN_MODE == "teaching":
    SIM_PATHS, SIM_T = 64, 1000
    TRAIN_STEPS = 5_000
elif RUN_MODE == "production":
    SIM_PATHS, SIM_T = 256, 5000
    TRAIN_STEPS = 20_000
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

print(f"RUN_MODE={RUN_MODE}: ergodic pool {SIM_PATHS}x{SIM_T}, {TRAIN_STEPS} training steps")

## **I. Economic environment**

We work in continuous time with an infinite horizon. The economy is populated by heterogeneous workers and heterogeneous firms. A worker has type $x \in \mathcal X$, and a firm has type $y \in \mathcal Y$.

The aggregate state of the economy is denoted by $z_t \in \mathcal Z$. It follows a continuous-time Markov chain with transition intensities
$
\lambda(z,\check{z}).
$

In the Section 3 application, $\mathcal Z$ contains three aggregate states: an expansion state, a normal recession state, and a disaster state.

A worker can be either unemployed ($u$) or employed ($e$) in a match. A firm can be either vacant ($v$) or producing ($p$) in a match. If a worker of type $x$ is matched with a firm of type $y$, the match produces flow output
$
F(x,y,z).
$

An unmatched worker receives flow value $b$, while a vacant firm pays flow cost $c$. Matches are destroyed at the exogenous separation rate
$
\delta(x,y,z).
$

The central endogenous state variable is the match distribution

$$
g_t(x,y),
$$

which describes the mass of active matches between workers of type $x$ and firms of type $y$. From $g_t$, we obtain the mass of employed workers and producing firms:

$$
g_t^e(x) = \int_{\mathcal Y} g_t(x,y)dy,
\qquad
g_t^p(y) = \int_{\mathcal X} g_t(x,y)dx.
$$

Given the total worker distribution $g^w(x)$ and the total firm distribution $g_t^f(y)$, the unmatched worker and vacant firm distributions are

$$
g_t^u(x) = g^w(x) - g_t^e(x),
\qquad
g_t^v(y) = g_t^f(y) - g_t^p(y).
$$

Aggregate unemployment, aggregate employment, aggregate vacant firms, and aggregate producing firms are therefore

$$
U_t = \int_{\mathcal X} g_t^u(x)dx,
\qquad
E_t = \int_{\mathcal X} g_t^e(x)dx,
\qquad
V_t = \int_{\mathcal Y} g_t^v(y)dy,
\qquad
P_t = \int_{\mathcal Y} g_t^p(y)dy.
$$

Unmatched workers and vacant firms meet through a matching function

$$
m(U_t,V_t).
$$

The meeting rate for an unmatched worker and a vacant firm are

$$
M_t^u = \frac{m(U_t,V_t)}{U_t},
\qquad
M_t^v = \frac{m(U_t,V_t)}{V_t}.
$$
respectively.

When a worker and firm meet, they decide whether to form a match. This decision is summarized by the acceptance rule

$$
\alpha(x,y,z,g) \in [0,1].
$$

Thus, the state of the economy is not only the aggregate shock $z$, but the pair

$$
(z,g).
$$

That is, the distribution $g$ affects who is available to meet, which changes matching decisions, which in turn changes the future distribution.


## **II. Recursive characterization of equilibrium**

The recursive state is $(z,g)$, where $z$ is the aggregate state and $g$ is the match distribution.

Let $V^u$ and $V^e$ denote the values of unmatched and employed workers, and let $V^v$ and $V^p$ denote the values of vacant and producing firms. The match surplus is

$$
S(x,y,z,g)
:=
V^p(x,y,z,g)-V^v(y,z,g)
+
V^e(x,y,z,g)-V^u(x,z,g).
$$

Surplus $S(x,y,z,g)$ is divided between workers and firms according to Nash Bargaining protocol with the bargaining parameter $\beta$. Thus,

$$
\begin{aligned}
	\beta S(x,y,z,g) ={}& V^e(x,y,z,g) - V^u(x,z,g) \\
	(1-\beta) S(x,y,z,g) ={}& V^p(x,y,z,g) - V^v(y,z,g).
\end{aligned}
$$

A meeting is accepted when the surplus is positive. In the exact model,

$$
\alpha(x,y,z,g)=\mathbf 1_{S(x,y,z,g)\geq 0}.
$$

In the numerical implementation, we use the smooth approximation

$$
\alpha(x,y,z,g)
=
\frac{1}{1+e^{-\xi S(x,y,z,g)}} \qquad \Rightarrow \qquad \lim_{\xi\rightarrow \infty}\alpha(x,y,z,g)=\mathbf 1_{S(x,y,z,g)\geq 0} \ \ \forall \ \  S(x,y,z,g)\neq0.
$$

The match distribution evolves according to

$$dg_t(x,y) =\mu_t^g(x,y,z,g) dt - \sum_{\check{z}_t \ne z_t} \sigma(z_t, \check{z}_t) g_t(x,y) dN(z_t;\check{z}_t),$$
where
$$
\mu^g(x,y,z,g)
=
-\big(\delta(x,y,z)+\varsigma(z)\big)g(x,y)
+
\alpha(x,y,z,g)m(U,V)
\frac{g^u(x)}{U}
\frac{g^v(y)}{V}.
$$

The first term removes matches through separations and firm exit. The second term adds newly accepted matches. When the aggregate state jumps from $z$ to $\check{z}$, a fraction $\sigma(z,\check{z})$ of matches is destroyed, so the post-jump distribution is

$$\bigl(1-\sigma(z,\check{z})\bigr)g.$$

In equilibrium, agents correctly forecast the law of motion of $g$. This means that the drift and jump they use in their value functions must equal the equilibrium the drift $\mu^g$ and jump $-\sigma(z_t, \check{z}_t) g_t(x,y)$. Therefore, the recursive equilibrium can be characterized directly through the surplus function $S(x,y,z,g)$.




## **III. Agent Hamilton-Jacobi-Bellman (HJB) Equations**

**Workers:**
Given beliefs and optimal decisions, the worker value functions $V^u$ and $V^e$ satisfy the Hamilton Jacobi Bellman (HJB) Equations: 
\begin{align}
\rho V^u(x,z,g) 
	={}& b + M^u \int \alpha(x,\tilde{y},z,g) 
    (V^e(x,\tilde{y},z,g) - V^u(x,z,g))
    \frac{g^v(\tilde{y})}{V}  d\tilde{y} \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z} \ne z} \lambda(z,\check{z})\left(V^u(x,\check{z}, (1-\sigma(z,\check{z}))g) - V^u(x,z,g)\right) + \langle D_{g} V^u, \breve{\mu}^g \rangle
    \\
%%
\rho V^e(x,y,z,g)
	={}&  \nonumber w(x,y,z,g) + \delta(x,y,z) (V^u(x,z,g) - V^e(x,y,z,g)) \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z}\ne z} \lambda(z,\check{z}) \left(V^e(x, y, \check{z},(1-\sigma(z,\check{z}))g) - V^e(x,y,z,g)\right) + \langle D_{g} V^e, \breve{\mu}^g \rangle
% \rho V^e(x,y,z,g)
% 	={}&  \nonumber w(x,y,z,g) + (\delta(x,y,z) + \varsigma(z)) (V^u(x,z,g) - V^e(x,y,z,g)) \nonumber \\
%     {}& \hspace{-1.5cm} + \sum_{\check{z}\ne z} \lambda(z,\check{z})\big[(1-\sigma(z,\check{z}))\left(V^e(x, y, \check{z},(1-\sigma(z,\check{z}))g) - V^e(x,y,z,g)\right) \nonumber \\
%     {}& \hspace{-1.5cm} + \sigma(z,\check{z})\left(V^u(x,\check{z}, (1-\sigma(z,\check{z}))g) - V^e(x,y,z,g)\right)\big] + \langle D_{g} V^e, \breve{\mu}^g \rangle
%     \label{eq:general:hjbes:Ve}
\end{align}



**Firms:**
Given beliefs and optimal decisions, the firm value functions $V^v$ and $V^p$ satisfy the Hamilton Jacobi Bellman (HJB) Equations: 
\begin{align}
%%
\rho V^v(y,z,g)
	={}& - c + M^v \int \alpha(\tilde{x},y,z,g) 
    (V^p(\tilde{x},y,z,g) - V^v(y,z,g)) \frac{g^u(\tilde{x})}{U} d\tilde{x} \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z} \ne z} \lambda(z,\check{z})\left(V^v(y,\check{z},(1-\sigma(z,\check{z}))g) - V^v(y,z,g)\right) + \left\langle D_{g} V^v, \breve{\mu}^g \right\rangle
    \\
%%
\rho V^p(x,y,z,g)
	={}& F(x,y,z) - w(x,y,z,g) + \delta(x,y,z) (V^v(y,z,g) - V^p(x,y,z,g)) \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z} \ne z} \lambda(z,\check{z}) (V^p(x, y,\check{z},(1-\sigma(z,\check{z}))g) - V^p(x, y, z, g)) + \left\langle D_{g} V^p, \breve{\mu}^g \right\rangle 
\end{align}

## **IV. Free-entry condition**

Free entry requires the expected value of a vacancy to be zero:

$$
0
=
\mathbb E_{\widetilde y}
\left[
V^v(\widetilde y,z,g)
\right]
=
\int_0^1
V^v(\widetilde y,z,g)\,\bar h(\widetilde y)\,d\widetilde y .
$$

Combining free entry with the vacancy HJB equation and the surplus-sharing rule gives

$$
\frac{m(U_t,V_t)}{V_t}
=
\frac{
c
}{
\displaystyle
\int
\int
\alpha(\widetilde x,\widetilde y,z_t,g_t)
\frac{g_t^u(\widetilde x)}{U_t}
(1-\beta)
S(\widetilde x,\widetilde y,z_t,g_t)
\,d\widetilde x\,d\widetilde y
}.
$$

Since the matching function is homothetic, this equation pins down the vacancy mass $V_t$. With uniform firm entry draws, the total firm distribution is

$$
g_t^f(y)=V_t+P_t,
$$

where

$$
P_t=\int_{\mathcal Y}g_t^p(y)\,dy .
$$

## **V. Master equation**

Combining the worker and firm HJB equations gives one master equation for the surplus:

$$
\begin{aligned}
0 = \mathcal L^S_S :=\;&
-\rho S(x,y,z,g)
+ F(x,y,z)
- \delta(x,y,z)S(x,y,z,g)
- b
\\
&-
\beta \frac{m(U,V)}{U}
\int_{\mathcal Y}
\alpha(x,\widetilde y,z,g)
S(x,\widetilde y,z,g)
\frac{g^v(\widetilde y)}{V}
\,d\widetilde y
\\
&+
c
-
(1-\beta)\frac{m(U,V)}{V}
\int_{\mathcal X}
\alpha(\widetilde x,y,z,g)
S(\widetilde x,y,z,g)
\frac{g^u(\widetilde x)}{U}
\,d\widetilde x
\\
&+
\sum_{\widehat z \neq z}
\lambda(z,\widehat z)
\left[
S\!\left(x,y,\widehat z,(1-\sigma(z,\widehat z))g\right)
-
S(x,y,z,g)
\right]
\\
&+
\left\langle D_g S,\mu^g \right\rangle .
\end{aligned}
$$

Using the master equation, $\alpha$ definition, KFE, "free-entry" condition, and beliefs consistency, DeepSAM approximates $S(x,y,z,g)$ with a neural network. Once $S$ is known, we recover the acceptance rule $\alpha$ and simulate the evolution of $g$.

## Calibration and the deterministic steady states

The calibration and training settings all live in `config/config.yaml`. We load it with
OmegaConf and pass it straight to `Train_NN`, so the object is simply built from that
dictionary.

`solve_steady_state()` then solves the model's **deterministic** steady state once for each
aggregate state $z \in \{L, H, D\}$: low separation (good times), high separation (bad
times), and the disaster state that stands in for COVID. These are fixed points of the
matching problem with the aggregate state frozen. They anchor the aggregate-risk solution,
and the unemployment rates they imply are the first thing to sanity-check against the
calibration.

In [ ]:
cfg = OmegaConf.load(ROOT / "config" / "config.yaml")
params = {
    k: v for k, v in OmegaConf.to_container(cfg.train_nn, resolve=True).items()
    if k != "_target_"
}

seed = int(cfg.seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ct = Train_NN(**params)
print(f"device {ct.device} | {ct.nx} worker types x {ct.ny} firm types")

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):     # the solver is chatty; keep the summary
    ct.solve_steady_state()
print(f"solve_steady_state: {time.monotonic() - t0:.1f}s")

gm_ss = np.load("gm_ss.npy")
gm_low = np.load("gm_low_delta.npy")
gm_high = np.load("gm_high_delta.npy")
gm_dis = np.load("gm_dis_delta.npy")

# State convention (see env.py): z = 0 is L, the good state (separation delta_0 - d_delta),
# z = 1 is H, the bad state (delta_0 + d_delta), z = 2 is D, the disaster state.
for label, gm in [("baseline", gm_ss), ("low separation (L)", gm_low),
                  ("high separation (H)", gm_high), ("disaster (D)", gm_dis)]:
    u = (ct.gw.mean() - np.mean(gm)).cpu().numpy() * 100
    print(f"  unemployment rate, {label:<22s}: {u:6.3f}%")

## The three elements of DeepSAM

The lecture describes every deep-learning solution method by three choices: *what* is
parameterized by a neural network, *which* objective its parameters minimize, and *where*
that objective is evaluated. The next three sections are those choices for DeepSAM, in code.
Each section copies the relevant piece of `src/train_nn.py` into the notebook so it can be
read line by line; the checks at the end of each section confirm the copy is the real
thing.

### Element 1: parameterize the surplus with a neural network

The one object DeepSAM learns is the **match surplus** $S(x, y, z, g)$: the value of a match
between worker type $x$ and firm type $y$, given the aggregate state $z$ *and the entire
cross-sectional distribution* $g$ of existing matches. That last argument is what makes the
problem high-dimensional: $g$ lives in $\mathbb{R}^{n_x \times n_y}$, 55 dimensions here,
which is why the network takes it directly as an input rather than summarising it.

Write $\omega = (x, y, z, g) \in \mathbb{R}^{3 + n_x n_y}$. The network is the feed-forward
form from the lecture,

$$
h^{(1)} = \phi\big(W^{(1)} \omega + b^{(1)}\big), \qquad
h^{(p)} = \phi\big(W^{(p)} h^{(p-1)} + b^{(p)}\big), \quad p = 2, \dots, H, \qquad
\widehat{S}(\omega; \Theta) = W^{(H+1)} h^{(H)} + b^{(H+1)},
$$

with $H = 4$ hidden layers of width 50, $\phi = \tanh$, and a linear output (the surplus can
have either sign). The parameters $\Theta = \{W^{(p)}, b^{(p)}\}$ are the unknowns. The class
below is copied verbatim from `src/train_nn.py`.

In [ ]:
class Master_PINN_S(torch.nn.Module):          # copied verbatim from src/train_nn.py
    def __init__(self,
            nn_width        = 50,
            nn_num_layers   = 4,
            n_x             = 2,
            n_y             = 2
            ):
        super(Master_PINN_S, self).__init__()
        # Construct an array of affine and activation functions.
        # Hidden layers:
        #   These are the hidden layers 1,2,..., nn_num_layers
        #   layers = [affine1, activation1, affine2, activation2, ...]
        layers = [torch.nn.Linear(n_x*n_y + 3, nn_width),torch.nn.Tanh()]
        for i in range(1,nn_num_layers):
            layers.append(torch.nn.Linear(nn_width, nn_width))
            layers.append(torch.nn.Tanh())

        layers.append(torch.nn.Linear(nn_width, 1))
        # Sequentially execute the affine and activations function;
        # This constructs the neural network approximation.
        self.net = torch.nn.Sequential(*layers)
        for i in range (0,nn_num_layers):
            # Apply the xavier normalization at the affine layers.
            torch.nn.init.xavier_normal_(self.net[2*i].weight)

    def forward(self, X):
        return self.net(X)


pinn_S = Master_PINN_S(
    nn_width=ct.nn_width, nn_num_layers=ct.nn_num_layers, n_x=ct.nx, n_y=ct.ny,
).to(ct.device).float()
print(pinn_S)

n_par = sum(p.numel() for p in pinn_S.parameters())
print(f"\n{n_par:,} parameters; input dimension 1 (x) + 1 (y) + 1 (z) + {ct.nx * ct.ny} (g) "
      f"= {3 + ct.nx * ct.ny}")

# One evaluation at the baseline steady state: worker type x, firm type y, state z = 0 (L),
# and the flattened steady-state distribution g. An untrained network returns numbers of
# no economic meaning yet; the point is only the shape of the input and output.
omega = torch.cat([
    torch.tensor([[ct.types_x[2], ct.types_y[5], 0.0]], dtype=torch.float32),
    torch.tensor(gm_ss, dtype=torch.float32).reshape(1, -1),
], dim=1).to(ct.device)
print(f"input omega: shape {tuple(omega.shape)}  ->  S_hat(omega) = {pinn_S(omega).item():+.4f}")

In [ ]:
# Load the converged parameters from the paper into the class defined in this notebook.
# (The state_dict loads only because the architecture above is exactly the one that was
# trained -- this is the check that the copy is the real thing.)
ckpt = torch.load(ROOT / "checkpoints" / "section3_surplus_best.pt", map_location=ct.device)
pinn_S.load_state_dict(ckpt["model_state_dict"])
pinn_S.eval()
print(f"Trained network loaded. S_hat at the same omega: {pinn_S(omega).item():+.4f}")

# The surplus on the whole type grid at the baseline steady state: positive where matches
# are formed, negative where they are not. calculate_alphas evaluates the network on all
# n_x * n_y (x, y) pairs at once and also returns the acceptance probability
# alpha = 1 / (1 + exp(-xi S)).
with torch.no_grad():
    S_grid, alpha_grid = ct.calculate_alphas(
        pinn_S, torch.zeros(1, 1, device=ct.device),
        torch.tensor(gm_ss, dtype=torch.float32, device=ct.device).reshape(1, -1),
    )
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), dpi=120, constrained_layout=True)
for ax, arr, title in [(axes[0], S_grid[0].cpu().numpy(), "surplus $S(x, y, L, g_{ss})$"),
                       (axes[1], alpha_grid[0].cpu().numpy(), r"acceptance $\alpha = \sigma(\xi S)$")]:
    im = ax.imshow(arr.T, origin="lower", aspect="auto",
                   extent=[ct.types_x[0], ct.types_x[-1], ct.types_y[0], ct.types_y[-1]])
    ax.set_xlabel("worker type $x$"); ax.set_ylabel("firm type $y$"); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

### Element 2: the objective, the master-equation residual

There is no target and no labelled data anywhere in DeepSAM. The network is fit to an
*equation*: at a state $\omega = (x, y, z, g)$, evaluate every term of the master equation of
Section V with $\widehat{S}$ in place of $S$, and call the result the residual
$\mathcal{L}^S \widehat{S}(\omega)$. The training loss is its mean square over a sample of
states,

$$
L(\Theta, Q) = \frac{1}{K} \sum_{k \le K} \big| \mathcal{L}^S \widehat{S}(\omega_k; \Theta) \big|^2 .
$$

The function below is the residual operator `S_pde_oper` from `src/train_nn.py`, copied
with comments that name each term of the equation. Three details of the implementation:

* **The distribution derivative.** $\langle D_g S, \mu^g \rangle$ needs
  $\partial \widehat{S} / \partial g_{ij}$ for all 55 entries of $g$. Automatic differentiation
  gives it in one backward pass (`torch.autograd.grad`), and because the residual is then
  differentiated *again* with respect to $\Theta$ during training, the graph is kept
  (`create_graph=True`).
* **The integrals** over $\tilde{x}$ and $\tilde{y}$ become sums over the type grids, so the
  operator needs $\widehat{S}$ and $\alpha$ on the *whole* grid at the current $(z, g)$, not
  only at the sampled $(x, y)$; `calculate_alphas` provides them.
* **The aggregate state** $z \in \{0, 1, 2\}$ is an index into $\delta(x, y, z)$, the flow
  exit rate $\varsigma(z)$ and the jump-exit matrix $\sigma(z, \hat z)$; productivity itself is
  $z_0 = 1$ in this calibration. The entry cost $c$ does not appear as a separate flow term
  here: it enters through the free-entry condition that pins down $V$ inside
  `marginals_U_V`.

In [ ]:
def master_equation_residual(ct, model_S, X):
    """The residual L^S S_hat at a batch of states X = [x, y, z, g_flat] (shape (N, 3 + n_x n_y)).
    Copied from Train_NN.S_pde_oper in src/train_nn.py; comments name the terms of Section V.
    Returns (residual (N, 1), alpha on the type grid (N, n_x, n_y), S_hat on the grid (N, n_x, n_y))."""
    device, n_x, n_y = ct.device, ct.nx, ct.ny
    rho, beta, b, z_0 = ct.rho, ct.beta, ct.b, ct.z_0
    delta_s, sigma_mat, varsigma_s = ct.delta_s, ct.sigma_mat, ct.varsigma_s
    tau, tau_w = ct.tau, ct.tau_w

    # ---- unpack the state: x, y (own types), z (aggregate state index), g (flattened distribution)
    g_m = X[:, 3:(n_x * n_y + 3)].clone().to(device)
    x = X[:, 0].clone().to(device).reshape(-1, 1)
    y = X[:, 1].clone().to(device).reshape(-1, 1)
    z = X[:, 2].clone().to(device).reshape(-1, 1)
    N = len(x)
    z_idx = z.view(-1).to(torch.long)

    types_x = torch.linspace(1 / n_x / 2, 1 - 1 / n_x / 2, n_x, device=device, dtype=torch.float32)
    types_y = torch.linspace(1 / n_y / 2, 1 - 1 / n_y / 2, n_y, device=device, dtype=torch.float32)

    def model_g(g_m_input):                 # S_hat at the same (x, y, z) but a different g
        X_temp = torch.clone(X)
        X_temp[:, 3:(n_x * n_y + 3)] = g_m_input
        return model_S(X_temp)

    def model_zg(z_in, g_m_input):          # S_hat at the same (x, y) but a different (z, g)
        X_temp = torch.clone(X)
        X_temp[:, 2:3] = z_in
        X_temp[:, 3:(n_x * n_y + 3)] = g_m_input
        return model_S(X_temp)

    # ---- S_hat at the sampled state, and dS_hat/dg by automatic differentiation
    S_batch = model_S(X).to(device)
    g_leaf = g_m.detach().clone().requires_grad_(True)
    Sg_pred = model_g(g_leaf).to(device)
    dS_dg = ct.get_derivs_1order(Sg_pred, g_leaf).to(device)          # (N, n_x n_y)

    # ---- S_hat and alpha = sigma(xi S_hat) on the whole type grid at (z, g): needed for the integrals
    S, alphas = ct.calculate_alphas(model_S, z, g_m)                   # (N, n_x, n_y) each

    # ---- marginals, unemployment U, vacancies V (free entry inside), vacancy marginal g_v
    g_e, g_u, U, g_p, g_v, V, g_f = ct.marginals_U_V(g_m, S, alphas, z)

    # ---- meeting rates M^v = m(U, V) / V and M^u = m(U, V) / U
    M_v = ct.m(U, V) / V
    M_u = ct.m(U, V) / U

    # ---- rows of the grid objects at the sampled worker type x and firm type y
    indices_x = (x / (types_x[1] - types_x[0]) - 0.5).long().flatten().to(device)
    indices_y = (y / (types_y[1] - types_y[0]) - 0.5).long().flatten().to(device)
    alphas_fixed_x = alphas[torch.arange(N), indices_x, :]             # alpha(x, . )
    alphas_fixed_y = alphas[torch.arange(N), :, indices_y]             # alpha( . , y)

    # ---- term 1:  -(rho + delta(x, y, z)) S  +  z_0 f(x, y)      [discounting, separation, output]
    term1 = -(rho + delta_s[z_idx, indices_x, indices_y].view(N, 1)) * S_batch + z_0 * ct.f_torch(x, y)

    # ---- term 2:  -(1 - beta) M^v (1/n_x) sum_x~ alpha(x~, y) S(x~, y) g^u(x~) / U      [firm-side integral]
    term2 = -(1 - beta) * M_v * (1 / n_x) * (1 / U) * \
        torch.sum(alphas_fixed_y * g_u * S[range(N), :, indices_y], dim=1, keepdim=True)

    # ---- term 3:  -b  -  beta M^u (1/n_y) sum_y~ alpha(x, y~) S(x, y~) g^v(y~) / V        [worker-side integral]
    term3 = -b - beta * M_u * (1 / n_y) * (1 / V) * \
        torch.sum(alphas_fixed_x * g_v * S[range(N), indices_x, :], dim=1, keepdim=True)

    # ---- term 4:  < D_g S, mu^g >   with the KFE drift mu^g from Section II
    term4 = torch.sum(dS_dg * ct.mu_g(g_m, M_u, V, alphas, g_e, g_p, g_f, z), dim=1, keepdim=True)

    # ---- term 5:  sum_{z^ != z} lambda(z, z^) [ S(x, y, z^, (1 - sigma(z, z^)) g) - S(x, y, z, g) ]
    # ---- term 6:  compensation of exiting matches; zero at tau = tau_w = 1 (full compensation)
    term5 = torch.zeros_like(S_batch)
    chi_tau = (tau - 1) * (1 - beta) + (tau_w - 1) * beta
    term6 = varsigma_s[z_idx].view(N, 1) * chi_tau * S_batch
    for z_prime in [0, 1, 2]:
        lam = (
            ct.lam_LH * (z == 0) * (z_prime == 1) + ct.lam_LD * (z == 0) * (z_prime == 2) +
            ct.lam_HL * (z == 1) * (z_prime == 0) + ct.lam_HD * (z == 1) * (z_prime == 2) +
            ct.lam_DL * (z == 2) * (z_prime == 0) + ct.lam_DH * (z == 2) * (z_prime == 1)
        ).to(dtype=S_batch.dtype)
        if torch.any(lam != 0):
            sigma_z_zp = sigma_mat[z_idx, z_prime].view(N, 1)
            g_plus = (1.0 - sigma_z_zp) * g_m                          # post-jump distribution
            z_prime_batch = torch.full((N, 1), float(z_prime), device=device, dtype=X.dtype)
            S_batch_switched_z = model_zg(z_prime_batch, g_plus).to(device)
            term5 += lam * (S_batch_switched_z - S_batch)
            term6 += lam * sigma_z_zp * chi_tau * S_batch_switched_z

    return term1 + term2 + term3 + term4 + term5 + term6, alphas, S_batch


print(inspect.signature(master_equation_residual))

In [ ]:
# Check: on a batch of states, the notebook's residual equals the library's S_pde_oper.
# (The batch comes from the mixed steady-state sampler of Element 3, explained next.)
S_batch_check, _, _ = ct.sample(256)
with torch.enable_grad():                    # dS/dg needs autograd even without backward()
    resid_nb, _, _ = master_equation_residual(ct, pinn_S, S_batch_check)
    resid_lib, _, _ = ct.S_pde_oper(pinn_S, S_batch_check)
assert torch.allclose(resid_nb, resid_lib, atol=1e-6, rtol=1e-5), "notebook residual differs from S_pde_oper"
print("notebook residual == library residual: OK")

loss = (resid_nb ** 2).mean()
print(f"trained network, 256 mixed-steady-state samples: mean squared residual = {loss.item():.3e}")
print("That mean square is the training loss -- there is no target anywhere in it.")

### Element 3: where to evaluate the objective, sampling the state

The residual is averaged over sample points $\omega_k = (x_k, y_k, z_k, g_k)$, so the third
choice is which states to sample. The own types $(x, y)$ and the aggregate state $z$ are low
dimensional: draw them from the type grids and the stationary distribution of the aggregate
chain. The hard part is the distribution $g$, 55 dimensions with a highly structured
support: the economy only ever visits distributions near the ones its own dynamics produce.
DeepSAM samples $g$ in phases (the sampling frame of the lecture):

* **Phase 1, mixtures of steady states.** Before there is a usable $\widehat{S}$, draw
  $g = \sum_s w_s\, g^s_{ss}$, a random convex combination of the three deterministic steady
  states, with the weight on the drawn aggregate state's own steady state
  $\varepsilon \sim \mathrm{Beta}(\alpha, 1)$ close to one. This puts mass "between" the
  steady states without simulating anything.
* **Phase 2, the ergodic set.** Once $\widehat{S}$ is reasonable, simulate the economy under
  it: draw paths of $z_t$, run the KFE for $g_t$ with the acceptance rule implied by
  $\widehat{S}$, discard a burn-in, and pool the visited $(z_t, g_t)$ pairs. Train on one part
  of the pool, validate on the other, and re-simulate as $\widehat{S}$ improves, because the
  ergodic set itself depends on the solution.
* **Phase 3, out of sample.** Report the loss on a freshly simulated evaluation set.

`sample` below is the Phase 1 sampler, copied from `src/train_nn.py`.

In [ ]:
def sample_mixed_steady_states(ct, N: int):      # copied from Train_NN.sample in src/train_nn.py
    """
    Draw N training samples with (x, y, z, g_m) where:
      - z ~ Categorical(self.prob_agg)
      - g_m is a convex combination of (gm_L, gm_H, gm_D)
        with a dominant weight on the chosen base state (z),
        using eps ~ Beta(alpha, 1) for the base and splitting the
        remainder across the two non-base states in proportion
        to their probabilities in self.prob_agg.
    """
    import torch.nn.functional as F
    device = ct.device
    n_x, n_y = ct.nx, ct.ny
    L = n_x * n_y

    # --- steady-state surfaces (L, H, D) ---
    gm_L = ct.gm_low.to(device)                                  # (n_x, n_y)
    gm_H = (ct.gm_low + ct.gm_high_low_diff).to(device)         # (n_x, n_y)
    gm_D = (ct.gm_low + ct.gm_dis_low_diff).to(device)          # (n_x, n_y)
    GM   = torch.stack([gm_L.flatten(), gm_H.flatten(), gm_D.flatten()], dim=0)  # (3, L)

    # --- sample z labels according to the stationary probabilities of the aggregate chain ---
    p = ct.prob_agg.to(device).flatten()
    p = p / p.sum()
    z = torch.multinomial(p, N, replacement=True).to(device)        # (N,)
    z_col = z.view(-1, 1)

    # --- dominant weight on the base state's steady state: eps ~ Beta(alpha, 1), i.e. eps = U^(1/alpha) ---
    alpha = getattr(ct, "alpha_base", 10.0)                          # larger => closer to the base steady state
    eps = torch.rand(N, 1, device=device) ** (1.0 / float(alpha))   # (N, 1) in (0, 1)
    one_hot = F.one_hot(z, num_classes=3).to(device=device, dtype=torch.float32)

    # --- the remaining weight 1 - eps is split across the other two states in proportion to p ---
    p_expand = p.unsqueeze(0).expand(N, 3)
    base_p   = (p_expand * one_hot).sum(dim=1, keepdim=True)
    rem      = 1.0 - eps
    share = p_expand.clone()
    share.scatter_(1, z_col, 0.0)
    denom = (1.0 - base_p).clamp_min(1e-12)
    share = share / denom

    W = eps * one_hot + rem * share                                  # convex weights, rows sum to 1
    g_m = W @ GM                                                     # (N, L): each g is a mixture

    # --- own types x, y uniformly over the grids ---
    idx_x = ct.unif_x.multinomial(N, replacement=True)
    x = ct.types_x[idx_x].reshape(N, 1).to(device)
    idx_y = ct.unif_y.multinomial(N, replacement=True)
    y = ct.types_y[idx_y].reshape(N, 1).to(device)

    z_feat = z_col.to(torch.float32)
    S_batch = torch.hstack((x, y, z_feat, g_m))                      # (N, 1 + 1 + 1 + L)
    return S_batch, W


torch.manual_seed(0)
S_phase1, W = sample_mixed_steady_states(ct, 2000)
print(f"batch shape {tuple(S_phase1.shape)} = (N, 1 + 1 + 1 + {ct.nx * ct.ny})")
z_draw = S_phase1[:, 2].long()
print("share of draws by aggregate state (L, H, D):",
      [f"{(z_draw == k).float().mean().item():.3f}" for k in range(3)],
      " target pi:", [f"{p:.3f}" for p in ct.prob_agg.tolist()])

fig, ax = plt.subplots(figsize=(6, 3.4), dpi=120)
ax.hist(W.max(dim=1).values.cpu().numpy(), bins=40, color="tab:blue")
ax.set_xlabel(r"weight on the drawn state's own steady state, $\varepsilon \sim$ Beta(10, 1)")
ax.set_ylabel("count"); ax.set_title("Phase 1: how far the sampled $g$ sits from a steady state")
plt.show()

**Phase 2, the ergodic sampler.** `build_ergodic_dataloaders` does three things:
`generate_z_paths` draws `SIM_PATHS` paths of the aggregate chain for `SIM_T` steps of
$dt = 0.01$; `simulate_economy` runs the KFE for $g_t$ along each path under the current
$\widehat{S}$ (Euler steps for the drift $\mu^g$, and the jump $(1 - \sigma(z, \hat z)) g$
whenever $z$ switches), recording $(z_t, g_t)$ every other step after a burn-in; the recorded
pairs are pooled, split into training and evaluation sets, and wrapped in data loaders that
attach a fresh uniform draw of $(x, y)$ to each $g$ every time a batch is formed. The
simulation is the same law of motion you will code yourself in notebook 02.

In [ ]:
# A few simulated paths, to see what the sampler records: the aggregate state and the
# unemployment rate it induces through the distribution.
z_demo = ct.generate_z_paths(N=4, T=SIM_T, dt=0.01, init_probs=ct.prob_agg, seed=1, as_float=False)
with contextlib.redirect_stdout(io.StringIO()):
    demo = ct.simulate_economy(pinn_S, z_demo, dt=0.01, substeps=5, g0=ct.gm_ss,
                               record_interval=2, burn_in_frac=0.0, return_paths=True)
t_rec = 0.01 * np.array(demo["record_idx"])
fig, axes = plt.subplots(2, 1, figsize=(8, 4.6), dpi=120, sharex=True, constrained_layout=True)
for i in range(4):
    axes[0].step(t_rec, demo["z_recorded"][i].cpu().numpy(), where="post", lw=1.2)
    axes[1].plot(t_rec, 100 * demo["U_paths"][i].cpu().numpy(), lw=1.2)
axes[0].set_yticks([0, 1, 2]); axes[0].set_yticklabels(["L", "H", "D"]); axes[0].set_ylabel("aggregate state $z_t$")
axes[1].set_ylabel("unemployment rate (%)"); axes[1].set_xlabel("years")
axes[0].set_title("Four simulated paths of the economy under the trained $\\widehat{S}$")
plt.show()

# The actual pool used for training and evaluation.
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    res = ct.build_ergodic_dataloaders(
        pinn_S, N_paths=SIM_PATHS, T=SIM_T, record_interval=2,
        batch_size_train=512, num_workers=0, seed=0,
    )
train_loader, eval_loader = res["train_loader"], res["eval_loader"]
print(f"ergodic simulation: {time.monotonic() - t0:.1f}s")
print(f"  train pool {len(train_loader.dataset):,} states | eval pool {len(eval_loader.dataset):,} states")

S_batch = next(iter(eval_loader))[0].to(ct.device)
print(f"  one batch: {tuple(S_batch.shape)}")

In [ ]:
# Where you evaluate the objective matters: the trained network was fit on the ergodic set,
# so its residual is smaller there than on Phase-1 mixtures, which include distributions the
# economy never visits.
with torch.enable_grad():
    r_p1, _, _ = master_equation_residual(ct, pinn_S, S_phase1[:512])
    r_erg, _, _ = master_equation_residual(ct, pinn_S, S_batch)
print(f"mean squared residual, trained network:")
print(f"  Phase-1 mixed steady-state samples : {(r_p1 ** 2).mean().item():.3e}")
print(f"  Phase-2 ergodic samples            : {(r_erg ** 2).mean().item():.3e}")

## Putting the three elements together: training

With the network (Element 1), the loss (Element 2) and the sampler (Element 3) in hand,
training is stochastic gradient descent on the mean squared residual over minibatches from
the ergodic pool, keeping the parameters with the best evaluation loss. That is all
`train_with_ergodic_loaders` does. Two runs at the same budget of `TRAIN_STEPS` steps:

1. **Continue from the checkpoint** at a low learning rate: it starts converged and moves
   little.
2. **From scratch.** A fresh network first gets the cheap homotopy initialisation (a
   supervised fit to the steady-state surplus, on Phase-1 samples) and then the same number
   of residual-minimizing steps. Comparing the two shows what the published training run
   bought.

In [ ]:
model_ft = Master_PINN_S(
    nn_width=ct.nn_width, nn_num_layers=ct.nn_num_layers, n_x=ct.nx, n_y=ct.ny
).to(ct.device).float()
model_ft.load_state_dict(ckpt["model_state_dict"])

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    out_ft = ct.train_with_ergodic_loaders(
        model_ft, optim.Adam(model_ft.parameters(), lr=1e-5),
        train_loader, eval_loader,
        total_steps=TRAIN_STEPS, eval_every=max(1, TRAIN_STEPS // 5),
        print_every=max(1, TRAIN_STEPS // 5), max_eval_batches=4,
    )
dt_ft = time.monotonic() - t0
print(f"continued from the checkpoint: {TRAIN_STEPS} steps in {dt_ft:.1f}s "
      f"({dt_ft / TRAIN_STEPS * 1000:.0f} ms/step), best eval loss {out_ft['best_eval_loss']:.3e}")

In [ ]:
model_new = Master_PINN_S(
    nn_width=ct.nn_width, nn_num_layers=ct.nn_num_layers, n_x=ct.nx, n_y=ct.ny
).to(ct.device).float()

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    ct.initial_guess(model_new, optim.Adam(model_new.parameters(), lr=ct.lr_init),
                     epochs=min(2000, 4 * TRAIN_STEPS), option="S")        # homotopy initialisation
    out_new = ct.train_with_ergodic_loaders(
        model_new, optim.Adam(model_new.parameters(), lr=1e-4),
        train_loader, eval_loader,
        total_steps=TRAIN_STEPS, eval_every=max(1, TRAIN_STEPS // 5),
        print_every=max(1, TRAIN_STEPS // 5), max_eval_batches=4,
    )
print(f"from scratch: initialisation + {TRAIN_STEPS} steps in {time.monotonic() - t0:.1f}s, "
      f"best eval loss {out_new['best_eval_loss']:.3e}")

fig, ax = plt.subplots(figsize=(7.5, 4.4), dpi=120)
for out, label, style in [(out_ft, "continued from the checkpoint", "-"),
                          (out_new, f"from scratch ({TRAIN_STEPS} steps)", "--")]:
    log = out["eval_log"]
    ax.semilogy(log["steps"], log["loss"], style, linewidth=2, marker="o", markersize=4, label=label)
ax.set_xlabel("gradient step"); ax.set_ylabel("evaluation loss (mean squared residual)")
ax.set_title("What a short training budget buys"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

gap = out_new["best_eval_loss"] / out_ft["best_eval_loss"]
print(f"after {TRAIN_STEPS} steps the fresh network is {gap:,.0f}x further from solving the "
      f"master equation than the shipped checkpoint")
print("The published pipeline: homotopy initialisation, a main phase to a loss threshold, "
      "then 8 rounds of 100,000 steps on rebuilt ergodic pools.")
assert np.isfinite(gap) and gap > 1.0, "the fresh network should not beat the checkpoint"

## How well is the master equation satisfied?

With no closed form and no coarse benchmark to compare against, the check on a solution is
that the equation it is supposed to satisfy holds everywhere the economy visits. Averaging
the squared residual of the trained network by $(x, y)$ over the evaluation pool shows
*where* in type space the solution is weakest, which a single scalar loss hides. The
absolute level carries the units of $S$ and depends on the evaluation pool: at `smoke` the
short burn-in still includes states some distance from the ergodic set.

In [ ]:
S_eval_list = plotting.freeze_S_batches(eval_loader, seed=1234)
maps_eval = plotting.compute_statewise_loss_maps_from_Slist(S_eval_list, ct, pinn_S)
E_all, extent = maps_eval["E_all"], maps_eval["extent"]

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4.4), constrained_layout=True, dpi=120)
im = ax.imshow(np.sqrt(E_all.T), origin="lower", aspect="auto", extent=extent)
ax.set_title("Master-equation residual (RMSE), trained network", fontsize=12)
ax.set_xlabel("worker type $x_i$"); ax.set_ylabel("firm type $y_j$")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, format=mticker.FormatStrFormatter("%.1e"))
plt.show()

rmse = float(np.sqrt(np.nanmean(E_all)))
print(f"overall RMSE of the master-equation residual: {rmse:.3e}")
print(f"worst cell: {np.sqrt(np.nanmax(E_all)):.3e}   best cell: {np.sqrt(np.nanmin(E_all)):.3e}")
assert np.isfinite(rmse) and rmse < 1.0, "residual is not a converged solution -- is the checkpoint loaded?"

## Summary

* The state of this economy is $(z, g)$: an aggregate shock and a 55-dimensional
  distribution of matches. There is no way to put that on a grid.
* **Element 1.** DeepSAM parameterizes the surplus $S(x, y, z, g)$ as a neural network that
  takes $g$ as a direct input.
* **Element 2.** The objective is the mean squared residual of the master equation, with
  $\partial S / \partial g$ from automatic differentiation. No targets, no data.
* **Element 3.** The residual is evaluated on mixtures of steady states first and then on the
  ergodic set of the simulated economy, so accuracy is concentrated where the economy goes.
* Training is SGD on that loss. A short run gets the loss falling quickly and then stalls
  orders of magnitude short of the checkpoint: the last digits of accuracy are most of the
  compute.

## Takeaway

The residual map is the diagnostic worth internalising. With no closed form to compare
against, the check on the solution is that the equation holds, everywhere the economy
visits and not just on average. Notebook 02 puts the solved model to work on the COVID
experiment.